# 4주차 — **넘파이**(NumPy)와 **판다스**(Pandas) 핵심

## PART 2 — **데이터 분석 기초**(Data Analysis Fundamentals)

---

## 학습 목표

1. **넘파이**(NumPy)의 배열 연산과 **벡터화**(Vectorization)의 원리를 이해한다.
2. **판다스**(Pandas)의 **데이터프레임**(DataFrame)과 **시리즈**(Series) 개념을 익힌다.
3. CSV 파일을 불러오고 기본 탐색 메서드를 사용하여 데이터 구조를 파악한다.
4. **브로드캐스팅**(Broadcasting)의 원리를 이해하고 활용한다.
5. 실전 데이터(**에어비앤비**)로 **데이터 분석 파이프라인**을 경험한다.
6. **텐서**(Tensor)와 **신경망**(Neural Network) 기초 개념을 이해한다.

---

## 활용 데이터

**뉴욕 에어비앤비**(Airbnb NYC 2019) — Kaggle `AB_NYC_2019.csv`

| 항목 | 내용 |
|---|---|
| 행(rows) | 약 48,895개 |
| 열(columns) | 16개 (`id`, `name`, `host_name`, `neighbourhood_group`, `price`, `room_type` 등) |
| 출처 | Kaggle — New York City Airbnb Open Data |

---

# PART 1: **넘파이**(NumPy) 핵심

---

## 1.1 왜 **넘파이**(NumPy)인가? — `for`문의 한계

### ▶ 비유: 택배 배달

> **for문** = 택배 기사가 상자를 **하나씩** 들고 간다. 100만 개면 100만 번 왕복한다.
> **NumPy** = 대형 트럭에 100만 개를 **한꺼번에** 싣고 간다. 왕복 1번이다.

이것이 **벡터화 연산**(Vectorized Operation)의 본질이다. 데이터를 하나씩 처리하는 것이 아니라, **배열 전체를 한 번에** 처리한다.

### 속도 비교 실험

In [ ]:
import numpy as np
import time

# ── 100만 개의 데이터 준비 ──
data_list = list(range(1_000_000))
data_array = np.array(data_list)

# ── 방법 1: for문 ──
start = time.time()
result_for = [x * 2 for x in data_list]
time_for = time.time() - start
print(f"[for문] 소요 시간: {time_for:.4f}초")

# ── 방법 2: NumPy ──
start = time.time()
result_np = data_array * 2
time_np = time.time() - start
print(f"[NumPy] 소요 시간: {time_np:.4f}초")

# ── 비교 ──
speedup = time_for / time_np
print(f"[비교] NumPy가 약 {speedup:.0f}배 빠르다!")
print(f"[검증] 결과 동일 여부: {result_for[:5]} == {list(result_np[:5])}")

```
[for문] 소요 시간: 0.0821초
[NumPy] 소요 시간: 0.0012초
[비교] NumPy가 약 68배 빠르다!
[검증] 결과 동일 여부: [0, 2, 4, 6, 8] == [0, 2, 4, 6, 8]
```

```
시각화: for문 vs NumPy 연산 방식

┌─────────────────────────────────────────────────────┐
│  for문 (Python 루프)           │  NumPy (벡터 연산)  │
│                                │                     │
│  [1] → ×2 → [2]   ← 1번째    │  [1, 2, 3, ..., N]  │
│  [2] → ×2 → [4]   ← 2번째    │       ↓ ×2          │
│  [3] → ×2 → [6]   ← 3번째    │  [2, 4, 6, ..., 2N] │
│  ...                           │                     │
│  [N] → ×2 → [2N]  ← N번째    │  ← 단 1번!          │
│                                │                     │
│  총 N번 반복 (느림)            │  C언어로 한 번에    │
└─────────────────────────────────────────────────────┘
```

### → AI 연결: 왜 AI에서 NumPy가 중요한가?

> **딥러닝**(Deep Learning)의 모든 계산은 **행렬 곱셈**(Matrix Multiplication)이다.
> 이미지 1장 = 224×224×3 = **150,528개** 숫자이다.
> 이것을 for문으로 처리하면 학습에 **수백 년**이 걸린다.
> NumPy(그리고 GPU)를 사용하면 **수 시간**이면 끝난다.
> **벡터화는 AI의 심장이다.**

---

## 1.2 `ndarray` — **N차원 배열**(N-dimensional Array)

### ▶ 비유: 엑셀 시트의 진화

> **리스트**(List) = 메모장에 줄글로 쓴 데이터 (1차원)
> **ndarray** = 엑셀 시트 (2차원) → 여러 시트를 겹친 것 (3차원) → 무한 확장 가능

In [ ]:
import numpy as np

# ── 1차원 배열 (벡터) ──
vec = np.array([1, 2, 3, 4, 5])
print(f"[1D 배열] {vec}")
print(f"[형태] shape = {vec.shape}")        # (5,)
print(f"[차원] ndim = {vec.ndim}")           # 1
print(f"[타입] dtype = {vec.dtype}")         # int64
print(f"[크기] size = {vec.size}")           # 5
print()

# ── 2차원 배열 (행렬) ──
mat = np.array([[1, 2, 3],
                [4, 5, 6]])
print(f"[2D 배열]\n{mat}")
print(f"[형태] shape = {mat.shape}")         # (2, 3) → 2행 3열
print(f"[차원] ndim = {mat.ndim}")           # 2
print(f"[크기] size = {mat.size}")           # 6
print()

# ── 3차원 배열 (텐서) ──
tensor = np.array([[[1, 2], [3, 4]],
                   [[5, 6], [7, 8]]])
print(f"[3D 배열]\n{tensor}")
print(f"[형태] shape = {tensor.shape}")      # (2, 2, 2)
print(f"[차원] ndim = {tensor.ndim}")        # 3

```
시각화: 차원별 배열 구조

1차원 (벡터, Vector):
  ┌───┬───┬───┬───┬───┐
  │ 1 │ 2 │ 3 │ 4 │ 5 │     shape = (5,)
  └───┴───┴───┴───┴───┘

2차원 (행렬, Matrix):
  ┌───┬───┬───┐
  │ 1 │ 2 │ 3 │  ← 0번째 행
  ├───┼───┼───┤
  │ 4 │ 5 │ 6 │  ← 1번째 행
  └───┴───┴───┘
  shape = (2, 3)  → 2행 × 3열

3차원 (텐서, Tensor):
  ┌─────────┐   ┌─────────┐
  │ 1  │  2 │   │ 5  │  6 │
  ├────┼────┤   ├────┼────┤
  │ 3  │  4 │   │ 7  │  8 │
  └─────────┘   └─────────┘
   페이지 0        페이지 1
  shape = (2, 2, 2)  → 2페이지 × 2행 × 2열
```

### → AI 연결: 텐서(Tensor)란 무엇인가?

> **텐서**(Tensor)는 다차원 배열의 일반적인 이름이다.
> **파이토치**(PyTorch)에서는 `np.array` 대신 `torch.Tensor`를 사용한다.
> 사실 NumPy의 ndarray와 PyTorch의 Tensor는 거의 같은 것이다.
>
> | 차원 | 이름 | AI에서의 의미 |
> |---|---|---|
> | 0차원 | **스칼라**(Scalar) | 손실 값(Loss) 1개 |
> | 1차원 | **벡터**(Vector) | 한 데이터의 **특성**(Feature) 모음 |
> | 2차원 | **행렬**(Matrix) | 데이터셋 전체 (행=샘플, 열=특성) |
> | 3차원 | **텐서**(Tensor) | 컬러 이미지 (높이×너비×채널) |
> | 4차원 | | 이미지 배치 (배치×높이×너비×채널) |

---

## 1.3 배열 생성 함수

In [ ]:
# ── 다양한 생성 방법 ──

# 0으로 채운 배열
zeros = np.zeros((3, 4))
print(f"[zeros] shape={zeros.shape}\n{zeros}\n")

# 1로 채운 배열
ones = np.ones((2, 3))
print(f"[ones] shape={ones.shape}\n{ones}\n")

# 특정 값으로 채운 배열
full = np.full((2, 3), 7)
print(f"[full(7)] shape={full.shape}\n{full}\n")

# 단위 행렬 (대각선이 1)
eye = np.eye(3)
print(f"[eye(3)] 단위 행렬:\n{eye}\n")

# 연속 숫자
arange = np.arange(0, 10, 2)
print(f"[arange(0,10,2)] {arange}")

# 균등 간격
linspace = np.linspace(0, 1, 5)
print(f"[linspace(0,1,5)] {linspace}")

# 랜덤 배열
rand_uniform = np.random.rand(3, 3)
print(f"[random.rand(3,3)] 균일분포:\n{np.round(rand_uniform, 2)}\n")

rand_normal = np.random.randn(3, 3)
print(f"[random.randn(3,3)] 정규분포:\n{np.round(rand_normal, 2)}\n")

rand_int = np.random.randint(1, 100, size=(2, 5))
print(f"[random.randint(1,100)] 정수 랜덤:\n{rand_int}")

```
시각화: 배열 생성 함수 한눈에 보기

┌──────────────────────────────────────────────────┐
│  함수                │  결과               │ 용도  │
├──────────────────────┼─────────────────────┼───────┤
│  np.zeros((2,3))     │  [[0,0,0],[0,0,0]]  │ 초기화│
│  np.ones((2,3))      │  [[1,1,1],[1,1,1]]  │ 마스크│
│  np.full((2,3), 7)   │  [[7,7,7],[7,7,7]]  │ 채우기│
│  np.eye(3)           │  [[1,0,0],          │ 단위  │
│                      │   [0,1,0],          │ 행렬  │
│                      │   [0,0,1]]          │       │
│  np.arange(0,10,2)   │  [0,2,4,6,8]        │ 범위  │
│  np.linspace(0,1,5)  │  [0,.25,.5,.75,1]   │ 등분  │
│  np.random.rand(2,3) │  [[0.1,0.8,...]]     │ 랜덤  │
└──────────────────────────────────────────────────┘
```

### → AI 연결: 왜 랜덤 초기화가 중요한가?

> **신경망**(Neural Network)의 **가중치**(Weight)는 학습 전에 **랜덤으로 초기화**한다.
> `np.random.randn()`으로 정규분포 랜덤 값을 만드는 것이 바로 신경망의 첫 단계이다.
> 모든 가중치를 0으로 초기화하면? → 모든 뉴런이 같은 값을 출력한다 → **학습이 안 된다!**
> 이것을 **대칭 파괴**(Symmetry Breaking)라 한다.

---

## 1.4 **인덱싱**(Indexing)과 **슬라이싱**(Slicing)

### ▶ 비유: 도서관 책꽂이

> **인덱싱** = "3번 선반의 2번째 책을 가져와라" → 정확한 위치 지정
> **슬라이싱** = "3번 선반의 2번째부터 5번째 책까지 가져와라" → 범위 지정
> **불리언 인덱싱** = "두께가 300페이지 이상인 책만 가져와라" → 조건 지정

In [ ]:
# ── 1차원 인덱싱 & 슬라이싱 ──
arr = np.array([10, 20, 30, 40, 50, 60, 70, 80, 90, 100])
print(f"[원본 배열] {arr}")
print(f"[인덱싱] arr[0] = {arr[0]}")        # 10 (첫 번째)
print(f"[인덱싱] arr[-1] = {arr[-1]}")       # 100 (마지막)
print(f"[슬라이싱] arr[2:5] = {arr[2:5]}")   # [30, 40, 50]
print(f"[슬라이싱] arr[:3] = {arr[:3]}")     # [10, 20, 30]
print(f"[슬라이싱] arr[::2] = {arr[::2]}")   # [10, 30, 50, 70, 90] 짝수 인덱스
print()

# ── 2차원 인덱싱 ──
mat = np.array([[1, 2, 3, 4],
                [5, 6, 7, 8],
                [9, 10, 11, 12]])
print(f"[2D 배열]\n{mat}")
print(f"[인덱싱] mat[1, 2] = {mat[1, 2]}")          # 7 (1행 2열)
print(f"[행 슬라이싱] mat[0, :] = {mat[0, :]}")      # [1, 2, 3, 4] (0번 행 전체)
print(f"[열 슬라이싱] mat[:, 1] = {mat[:, 1]}")      # [2, 6, 10] (1번 열 전체)
print(f"[부분 행렬] mat[0:2, 1:3] =\n{mat[0:2, 1:3]}")  # [[2,3],[6,7]]

```
시각화: 2D 인덱싱 가이드

        열0  열1  열2  열3
       ┌────┬────┬────┬────┐
  행0  │  1 │  2 │  3 │  4 │   mat[0, :] = [1, 2, 3, 4]
       ├────┼────┼────┼────┤
  행1  │  5 │  6 │ [7]│  8 │   mat[1, 2] = 7
       ├────┼────┼────┼────┤
  행2  │  9 │ 10 │ 11 │ 12 │   mat[:, 1] = [2, 6, 10]
       └────┴────┴────┴────┘               ↑
                  ↑                     1번 열 전체
            mat[0:2, 1:3] = [[2, 3],
                             [6, 7]]
```

---

## 1.5 **불리언 인덱싱**(Boolean Indexing)

### ▶ 비유: 체로 거르기

> 쌀에서 돌을 골라내듯, **조건에 맞는 데이터만 걸러낸다.**
> 체의 구멍 크기 = 조건(condition)
> 체를 통과한 쌀 = 조건을 만족하는 데이터

In [ ]:
scores = np.array([85, 92, 78, 95, 88, 67, 91, 73, 98, 82])
print(f"[원본 점수] {scores}")
print(f"[조건 마스크] scores >= 90 → {scores >= 90}")
print(f"[결과] 90점 이상: {scores[scores >= 90]}")
print(f"[개수] 90점 이상: {np.sum(scores >= 90)}명")
print(f"[비율] 90점 이상: {np.mean(scores >= 90)*100:.1f}%")
print()

# ── 복합 조건 ──
print(f"[AND] 80~90점: {scores[(scores >= 80) & (scores <= 90)]}")
print(f"[OR]  80 미만 또는 95 초과: {scores[(scores < 80) | (scores > 95)]}")
print(f"[NOT] 90점 미만: {scores[~(scores >= 90)]}")

```
시각화: 불리언 인덱싱 과정

원본:   [85, 92, 78, 95, 88, 67, 91, 73, 98, 82]
         ↓   ↓   ↓   ↓   ↓   ↓   ↓   ↓   ↓   ↓
조건:    >= 90?
마스크:  [ F,  T,  F,  T,  F,  F,  T,  F,  T,  F]
         ✗   ✓   ✗   ✓   ✗   ✗   ✓   ✗   ✓   ✗
결과:       [92,     95,         91,     98]  ← 4개만 살아남음!
```

### → AI 연결: 불리언 마스크와 **어텐션**(Attention)

> **트랜스포머**(Transformer)의 **마스킹**(Masking)은 불리언 인덱싱의 확장이다.
> 예: ChatGPT가 문장을 생성할 때, **아직 생성하지 않은 단어**를 보지 못하도록
> 불리언 마스크로 가린다. 이것을 **인과적 마스킹**(Causal Masking)이라 한다.
>
> ```
> "나는 밥을 ___"
>  ✓   ✓   ✓   ✗  ← 미래 단어를 마스킹!
> ```

---

## 1.6 **벡터화 연산**(Vectorized Operations) 상세

In [ ]:
a = np.array([1, 2, 3, 4, 5])
b = np.array([10, 20, 30, 40, 50])

# ── 사칙연산 ──
print(f"[덧셈]    a + b  = {a + b}")       # [11, 22, 33, 44, 55]
print(f"[뺄셈]    a - b  = {a - b}")       # [-9, -18, -27, -36, -45]
print(f"[곱셈]    a * b  = {a * b}")       # [10, 40, 90, 160, 250]
print(f"[나눗셈]  a / b  = {a / b}")       # [0.1, 0.1, 0.1, 0.1, 0.1]
print(f"[거듭제곱] a ** 2 = {a ** 2}")      # [1, 4, 9, 16, 25]
print()

# ── 비교 연산 ──
print(f"[비교] a > 3  = {a > 3}")          # [F, F, F, T, T]
print(f"[비교] a == 3 = {a == 3}")         # [F, F, T, F, F]
print()

# ── 유니버설 함수 (ufunc) ──
print(f"[sqrt]  np.sqrt(a)  = {np.sqrt(a)}")
print(f"[exp]   np.exp(a)   = {np.exp(a)}")
print(f"[log]   np.log(a)   = {np.round(np.log(a), 3)}")
print(f"[abs]   np.abs([-3, -1, 2]) = {np.abs(np.array([-3, -1, 2]))}")

```
시각화: 벡터화 연산 = 원소별(element-wise) 연산

    a = [1,  2,  3,  4,  5]
    b = [10, 20, 30, 40, 50]
         ↓   ↓   ↓   ↓   ↓
a + b = [11, 22, 33, 44, 55]    ← 같은 위치끼리 더한다
a * b = [10, 40, 90,160,250]    ← 같은 위치끼리 곱한다

⚠️ 주의: 이것은 '행렬 곱셈'이 아니라 '원소별 곱셈'이다!
   행렬 곱셈은 np.dot() 또는 @ 연산자를 사용한다.
```

---

## 1.7 **브로드캐스팅**(Broadcasting)

### ▶ 비유: 빵에 버터 바르기

> 빵 8조각(배열)에 버터 1덩이(스칼라)를 바른다고 하자.
> 버터를 8등분해서 각 조각에 바르는 것이 아니라,
> **버터가 자동으로 8조각으로 복제되어** 각 빵에 바른다.
> 이것이 **브로드캐스팅**이다 — **작은 배열이 큰 배열에 맞게 자동 확장**된다.

In [ ]:
# ── 스칼라 + 배열 ──
arr = np.array([1, 2, 3, 4, 5])
print(f"[원본] {arr}")
print(f"[+10]  {arr + 10}")     # [11, 12, 13, 14, 15]
print(f"[×3]   {arr * 3}")      # [ 3,  6,  9, 12, 15]
print()

# ── 2D + 1D 브로드캐스팅 ──
mat = np.array([[1, 2, 3],
                [4, 5, 6],
                [7, 8, 9]])
row = np.array([10, 20, 30])
print(f"[행렬]\n{mat}")
print(f"[행 벡터] {row}")
print(f"[행렬 + 행 벡터]\n{mat + row}")
print()

# ── 열 벡터와의 브로드캐스팅 ──
col = np.array([[100], [200], [300]])
print(f"[열 벡터]\n{col}")
print(f"[행렬 + 열 벡터]\n{mat + col}")

```
시각화: 브로드캐스팅 과정 (2D + 1D)

행렬 (3×3):          행 벡터 (1×3):
┌───┬───┬───┐        ┌───┬───┬───┐
│ 1 │ 2 │ 3 │        │10 │20 │30 │  ← 원본 (1행)
├───┼───┼───┤    +   └───┴───┴───┘
│ 4 │ 5 │ 6 │             ↓ 자동 복제!
├───┼───┼───┤        ┌───┬───┬───┐
│ 7 │ 8 │ 9 │        │10 │20 │30 │
└───┴───┴───┘        ├───┼───┼───┤
                     │10 │20 │30 │  ← 3행으로 확장
                     ├───┼───┼───┤
                     │10 │20 │30 │
                     └───┴───┴───┘

결과 (3×3):
┌───┬───┬───┐
│11 │22 │33 │
├───┼───┼───┤
│14 │25 │36 │
├───┼───┼───┤
│17 │28 │39 │
└───┴───┴───┘
```

### 브로드캐스팅 규칙 3가지

```
규칙 1: 차원 수가 다르면, 작은 쪽의 shape 앞에 1을 추가한다.
         (3,3) + (3,) → (3,3) + (1,3)

규칙 2: 각 차원에서 크기가 1인 쪽이 다른 쪽에 맞게 확장된다.
         (3,3) + (1,3) → (3,3) + (3,3)

규칙 3: 크기가 1도 아니고 서로 다르면 → 오류(Error)!
         (3,3) + (2,) → ❌ 불가능!

브로드캐스팅 가능 여부 판단표:
┌──────────────┬──────────────┬────────┐
│  Shape A     │  Shape B     │ 가능?  │
├──────────────┼──────────────┼────────┤
│  (3, 3)      │  (3,)        │  ✅    │
│  (3, 3)      │  (1, 3)      │  ✅    │
│  (3, 3)      │  (3, 1)      │  ✅    │
│  (4, 3)      │  (3,)        │  ✅    │
│  (4, 3)      │  (4,)        │  ❌    │
│  (2, 3, 4)   │  (3, 4)      │  ✅    │
│  (2, 3, 4)   │  (2, 1, 4)   │  ✅    │
└──────────────┴──────────────┴────────┘
```

### → AI 연결: 브로드캐스팅과 **배치 처리**(Batch Processing)

> 신경망에서 데이터를 **배치**(Batch) 단위로 처리할 때 브로드캐스팅이 필수적이다.
> 예: 32개 이미지를 동시에 처리할 때, **편향**(Bias) 벡터 1개가 32개 이미지 모두에
> 브로드캐스팅되어 더해진다.
>
> ```
> 입력: (32, 784) ← 32개 이미지, 각 784 픽셀
> 가중치: (784, 128) ← 784→128 변환
> 편향: (128,) ← 128개 뉴런의 편향
>
> 출력 = 입력 @ 가중치 + 편향
>         (32, 128)   + (128,)  ← 브로드캐스팅!
>       = (32, 128)
> ```

---

## 1.8 **통계 함수**(Statistical Functions)

### ▶ 비유: 반 전체 시험 성적 분석

> 선생님이 30명의 수학 점수를 분석한다고 하자.
> **평균** = 전체 수준 파악, **표준편차** = 점수가 얼마나 퍼져있는지,
> **최솟값/최댓값** = 극단값 파악, **중앙값** = 이상치에 강건한 대표값

In [ ]:
scores = np.array([85, 92, 78, 95, 88, 67, 91, 73, 98, 82,
                   90, 76, 88, 94, 71, 86, 93, 79, 87, 84])
print(f"[원본 점수] {scores}")
print(f"[개수]   count    = {len(scores)}")
print(f"[합계]   sum      = {np.sum(scores)}")
print(f"[평균]   mean     = {np.mean(scores):.1f}")
print(f"[중앙값] median   = {np.median(scores):.1f}")
print(f"[표준편차] std    = {np.std(scores):.1f}")
print(f"[분산]   var      = {np.var(scores):.1f}")
print(f"[최솟값] min      = {np.min(scores)}")
print(f"[최댓값] max      = {np.max(scores)}")
print(f"[범위]   range    = {np.ptp(scores)}")  # max - min
print(f"[최솟값 위치] argmin = {np.argmin(scores)}")  # 인덱스 5 → 67
print(f"[최댓값 위치] argmax = {np.argmax(scores)}")  # 인덱스 8 → 98
print()

# ── 2D 배열에서 축(axis) 지정 ──
data = np.array([[85, 92, 78],
                 [95, 88, 67],
                 [91, 73, 98]])
print(f"[2D 데이터]\n{data}")
print(f"[전체 평균]       np.mean(data)     = {np.mean(data):.1f}")
print(f"[행별 평균] axis=1 np.mean(data, 1) = {np.mean(data, axis=1)}")  # 각 학생 평균
print(f"[열별 평균] axis=0 np.mean(data, 0) = {np.mean(data, axis=0)}")  # 각 과목 평균

```
시각화: axis의 의미 — "어떤 방향으로 접을 것인가?"

           과목0  과목1  과목2
         ┌──────┬──────┬──────┐
학생0    │  85  │  92  │  78  │  → axis=1: 행을 따라 → 학생별 평균 [85.0, 83.3, 87.3]
         ├──────┼──────┼──────┤
학생1    │  95  │  88  │  67  │
         ├──────┼──────┼──────┤
학생2    │  91  │  73  │  98  │
         └──────┴──────┴──────┘
            ↓      ↓      ↓
         axis=0: 열을 따라 ↓ → 과목별 평균 [90.3, 84.3, 81.0]

▶ 기억법: axis=0은 "행이 사라지는 방향" (행을 접는다)
          axis=1은 "열이 사라지는 방향" (열을 접는다)
```

### → AI 연결: **평균과 표준편차**로 데이터를 **정규화**(Normalization)한다

> 머신러닝에서 **특성 스케일링**(Feature Scaling)은 필수이다.
> 키(cm)와 몸무게(kg)의 단위가 다르면 모델이 혼란을 겪는다.
>
> **표준화**(Standardization) 공식:
> ```
> z = (x - mean) / std
> ```
> 이 한 줄이 NumPy로 이렇게 된다:
> ```python
> z = (data - np.mean(data, axis=0)) / np.std(data, axis=0)
> ```

---

## 1.9 **행렬 연산**(Matrix Operations)

In [ ]:
A = np.array([[1, 2],
              [3, 4]])
B = np.array([[5, 6],
              [7, 8]])

# ── 원소별 곱셈 vs 행렬 곱셈 ──
print(f"[원소별 곱셈] A * B =\n{A * B}")      # [[5,12],[21,32]]
print(f"[행렬 곱셈]   A @ B =\n{A @ B}")      # [[19,22],[43,50]]
print(f"[행렬 곱셈]   np.dot(A, B) =\n{np.dot(A, B)}")  # 동일
print()

# ── 전치 행렬 ──
print(f"[전치] A.T =\n{A.T}")                  # [[1,3],[2,4]]
print()

# ── 역행렬 ──
A_inv = np.linalg.inv(A)
print(f"[역행렬] A⁻¹ =\n{A_inv}")
print(f"[검증] A @ A⁻¹ ≈ 단위행렬:\n{np.round(A @ A_inv, 10)}")
print()

# ── 행렬식 ──
det = np.linalg.det(A)
print(f"[행렬식] det(A) = {det}")

```
시각화: 원소별 곱셈 vs 행렬 곱셈

원소별 곱셈 (Element-wise):     행렬 곱셈 (Matrix Multiplication):
[1, 2]   [5, 6]   [ 5, 12]    [1, 2]   [5, 6]   [1×5+2×7, 1×6+2×8]
[3, 4] * [7, 8] = [21, 32]    [3, 4] @ [7, 8] = [3×5+4×7, 3×6+4×8]
                                                = [  19,      22   ]
같은 위치끼리 곱한다              행×열의 내적(dot product)        [  43,      50   ]
```

### → AI 연결: 행렬 곱셈 = 신경망의 **순전파**(Forward Propagation)

> 신경망의 한 **층**(Layer)은 이렇게 계산된다:
> ```
> 출력 = 활성화함수(입력 @ 가중치 + 편향)
> ```
> 즉, **행렬 곱셈 한 번 = 한 층 계산**이다.
> 10층짜리 신경망은 행렬 곱셈을 10번 하는 것이다.

---

## 1.10 **메모리 구조**(Memory Layout) — NumPy가 빠른 이유

### ▶ 비유: 아파트 주소 vs 산속 오두막

> **리스트**: 원소가 메모리 여기저기에 흩어져 있다. (산속 오두막들)
> 각 원소를 찾으려면 매번 주소를 따라가야 한다 (포인터 추적).
>
> **NumPy 배열**: 원소가 메모리에 **연속적으로** 붙어 있다. (아파트 동)
> CPU가 한 번에 쭉 읽을 수 있다 (캐시 효율).

In [ ]:
import sys

# ── 메모리 사용량 비교 ──
py_list = list(range(1000))
np_array = np.arange(1000, dtype=np.int64)

list_size = sys.getsizeof(py_list) + sum(sys.getsizeof(x) for x in py_list)
array_size = np_array.nbytes

print(f"[Python 리스트] 1000개 정수: {list_size:,} bytes")
print(f"[NumPy 배열]   1000개 정수: {array_size:,} bytes")
print(f"[비교] 리스트가 약 {list_size/array_size:.1f}배 더 많은 메모리 사용")
print()

# ── dtype 확인 ──
print(f"[dtype 예시]")
print(f"  int32:   {np.array([1], dtype=np.int32).nbytes} bytes/원소")
print(f"  int64:   {np.array([1], dtype=np.int64).nbytes} bytes/원소")
print(f"  float32: {np.array([1.0], dtype=np.float32).nbytes} bytes/원소")
print(f"  float64: {np.array([1.0], dtype=np.float64).nbytes} bytes/원소")

```
시각화: Python 리스트 vs NumPy 배열의 메모리 구조

Python 리스트 (포인터 배열):
  ┌──────┐
  │ ptr0 │──→ [type: int, value: 1, refcount: 2] ← 각 원소가 객체
  ├──────┤
  │ ptr1 │──→ [type: int, value: 2, refcount: 1] ← 메모리 여기저기
  ├──────┤
  │ ptr2 │──→ [type: int, value: 3, refcount: 1]
  └──────┘
  ↑ 포인터 추적 필요 → 느림 + 메모리 낭비

NumPy ndarray (연속 메모리):
  ┌─────────────────────────────────────┐
  │ header (dtype, shape, strides, ...) │
  ├────┬────┬────┬────┬────┬────┬─ ... ─┤
  │ 1  │ 2  │ 3  │ 4  │ 5  │ 6  │ ...  │  ← 값만 연속 저장
  └────┴────┴────┴────┴────┴────┴─ ... ─┘
  ↑ CPU 캐시 라인에 딱 맞음 → 빠름 + 메모리 절약
```

---

## 1.11 **인터랙티브 위젯**: NumPy 연산 실험실

> ⚠️ 아래 코드는 **Google Colab**에서 실행하면 **슬라이더 위젯**으로 동작한다.

In [ ]:
# ── 위젯 1: 브로드캐스팅 시각화 ──
import ipywidgets as widgets
from IPython.display import display, clear_output

def broadcasting_demo(scalar):
    arr = np.array([1, 2, 3, 4, 5])
    result = arr * scalar
    clear_output(wait=True)
    print(f"┌─────────────────────────────────────────┐")
    print(f"│  브로드캐스팅 실험실                     │")
    print(f"├─────────────────────────────────────────┤")
    print(f"│  배열:  {arr}                      │")
    print(f"│  × {scalar:4.1f}  (스칼라 브로드캐스팅)          │")
    print(f"│  결과:  {result}  │")
    print(f"│                                         │")
    print(f"│  시각화:                                 │")
    for i in range(5):
        bar = "█" * int(result[i] / max(result) * 20)
        print(f"│  [{i}] {result[i]:6.1f} {bar:<20} │")
    print(f"└─────────────────────────────────────────┘")

slider = widgets.FloatSlider(value=2, min=0.5, max=10, step=0.5,
                             description='곱하기:')
widgets.interact(broadcasting_demo, scalar=slider)

In [ ]:
# ── 위젯 2: 벡터화 속도 비교 실험 ──
def speed_comparison(data_size):
    data_size = int(data_size)
    data = list(range(data_size))
    arr = np.array(data)
    
    # for문
    start = time.time()
    _ = [x * 2 for x in data]
    t_for = time.time() - start
    
    # NumPy
    start = time.time()
    _ = arr * 2
    t_np = time.time() - start
    
    speedup = t_for / t_np if t_np > 0 else float('inf')
    
    clear_output(wait=True)
    print(f"┌─────────────────────────────────────────┐")
    print(f"│  속도 비교 실험 (데이터 {data_size:,}개)    │")
    print(f"├─────────────────────────────────────────┤")
    print(f"│  for문:  {t_for:.6f}초                    │")
    print(f"│  NumPy:  {t_np:.6f}초                     │")
    print(f"│  속도비: NumPy가 약 {speedup:.0f}배 빠르다     │")
    print(f"│                                         │")
    bar_for = "█" * min(40, int(t_for * 1000))
    bar_np  = "█" * max(1, min(40, int(t_np * 1000)))
    print(f"│  for문 {bar_for}                       │")
    print(f"│  NumPy {bar_np}                         │")
    print(f"└─────────────────────────────────────────┘")

size_slider = widgets.IntSlider(value=100000, min=1000, max=5000000,
                                 step=100000, description='데이터 수:')
widgets.interact(speed_comparison, data_size=size_slider)

---

## 1.12 NumPy **연습문제** (기초)

### 문제 1. 배열 생성과 정보 확인

In [ ]:
# 1부터 20까지의 정수를 담은 1차원 배열을 만들고,
# 이를 4행 5열의 2차원 배열로 변환한 후,
# shape, ndim, dtype, size를 출력하라.

# 여기에 코드를 작성하라

### 문제 2. 불리언 인덱싱 활용

In [ ]:
# 다음 배열에서 짝수만 추출하고, 짝수의 합을 구하라.
data = np.array([15, 22, 7, 34, 11, 48, 5, 60, 33, 42])

# 여기에 코드를 작성하라

### 문제 3. 브로드캐스팅 활용

In [ ]:
# 3x4 행렬에서 각 열의 평균을 빼서 "중심화(centering)"를 수행하라.
# (각 열의 평균이 0이 되도록 변환)
mat = np.array([[10, 20, 30, 40],
                [50, 60, 70, 80],
                [90, 100, 110, 120]])

# 여기에 코드를 작성하라
# 힌트: 열별 평균 → axis=0으로 구한다

### 문제 4. 행렬 연산

In [ ]:
# 2x3 행렬 A와 3x2 행렬 B를 만들고,
# 행렬 곱셈(A @ B)을 수행한 후 결과의 shape를 확인하라.
# 또한 A @ B와 B @ A의 결과가 다름을 보여라.

# 여기에 코드를 작성하라

### 문제 5. 통계 함수 종합

In [ ]:
# 다음 2D 배열에서:
# (1) 전체 평균, (2) 행별 최댓값, (3) 열별 표준편차를 구하라.
data = np.array([[85, 92, 78, 95],
                 [88, 67, 91, 73],
                 [98, 82, 90, 76]])

# 여기에 코드를 작성하라

<details>
<summary>▶ NumPy 기초 연습문제 해답 (클릭하여 펼치기)</summary>

#### 해답 1

<pre><code class="python">arr = np.arange(1, 21)
print(f"[1D] {arr}")

mat = arr.reshape(4, 5)
print(f"[2D]\n{mat}")
print(f"[shape] {mat.shape}")   # (4, 5)
print(f"[ndim]  {mat.ndim}")    # 2
print(f"[dtype] {mat.dtype}")   # int64
print(f"[size]  {mat.size}")    # 20</code></pre>

#### 해답 2

<pre><code class="python">data = np.array([15, 22, 7, 34, 11, 48, 5, 60, 33, 42])
evens = data[data % 2 == 0]
print(f"[짝수만] {evens}")          # [22, 34, 48, 60, 42]
print(f"[짝수 합] {np.sum(evens)}") # 206</code></pre>

#### 해답 3

<pre><code class="python">mat = np.array([[10, 20, 30, 40],
                [50, 60, 70, 80],
                [90, 100, 110, 120]])
col_mean = np.mean(mat, axis=0)
print(f"[열별 평균] {col_mean}")        # [50, 60, 70, 80]
centered = mat - col_mean               # 브로드캐스팅!
print(f"[중심화 결과]\n{centered}")
print(f"[검증: 열별 평균 ≈ 0] {np.mean(centered, axis=0)}")</code></pre>

#### 해답 4

<pre><code class="python">A = np.array([[1, 2, 3], [4, 5, 6]])         # (2, 3)
B = np.array([[7, 8], [9, 10], [11, 12]])     # (3, 2)
AB = A @ B
BA = B @ A
print(f"[A @ B] shape={AB.shape}\n{AB}")      # (2, 2)
print(f"[B @ A] shape={BA.shape}\n{BA}")      # (3, 3)
print(f"[결론] shape도 값도 다르다! 행렬 곱셈은 교환법칙이 성립하지 않는다.")</code></pre>

#### 해답 5

<pre><code class="python">data = np.array([[85, 92, 78, 95],
                 [88, 67, 91, 73],
                 [98, 82, 90, 76]])
print(f"[전체 평균] {np.mean(data):.1f}")
print(f"[행별 최대] {np.max(data, axis=1)}")      # [95, 91, 98]
print(f"[열별 표준편차] {np.std(data, axis=0)}")</code></pre>

</details>

---

# PART 2: **판다스**(Pandas) 핵심

---

## 2.1 왜 **판다스**(Pandas)인가? — NumPy만으로는 부족한 이유

### ▶ 비유: 숫자만 있는 엑셀 vs 이름표가 붙은 엑셀

> **NumPy 배열** = 행과 열에 **이름이 없다.** 0번 열이 뭔지 기억해야 한다.
> **Pandas DataFrame** = 행과 열에 **이름표**(라벨)가 붙어 있다. `df['price']`처럼 직관적이다.

In [ ]:
import pandas as pd
import numpy as np

# ── NumPy로 데이터 다루기 (불편함) ──
data_np = np.array([['Alice', 25, 50000],
                    ['Bob', 30, 60000],
                    ['Carol', 28, 55000]])
print(f"[NumPy] 이름: {data_np[:, 0]}")     # 0번 열이 이름이라는 걸 기억해야 한다
print(f"[NumPy] 나이: {data_np[:, 1]}")     # 문자열이라 계산 불가!
print()

# ── Pandas로 데이터 다루기 (직관적) ──
df = pd.DataFrame({
    'name': ['Alice', 'Bob', 'Carol'],
    'age': [25, 30, 28],
    'salary': [50000, 60000, 55000]
})
print(f"[Pandas]\n{df}")
print(f"\n[이름으로 접근] df['age'] =\n{df['age']}")
print(f"\n[평균 나이] {df['age'].mean()}")     # 바로 계산 가능!
print(f"[최고 연봉] {df['salary'].max()}")

```
시각화: NumPy 배열 vs Pandas DataFrame

NumPy ndarray:                    Pandas DataFrame:
┌───────┬────┬───────┐           ┌───────┬─────┬────────┐
│ Alice │ 25 │ 50000 │           │ name  │ age │ salary │ ← 열 이름!
├───────┼────┼───────┤           ├───────┼─────┼────────┤
│  Bob  │ 30 │ 60000 │           │ Alice │  25 │  50000 │ ← 행 인덱스 0
├───────┼────┼───────┤           ├───────┼─────┼────────┤
│ Carol │ 28 │ 55000 │           │  Bob  │  30 │  60000 │ ← 행 인덱스 1
└───────┴────┴───────┘           ├───────┼─────┼────────┤
                                 │ Carol │  28 │  55000 │ ← 행 인덱스 2
  → "0번 열이 뭐였지?"           └───────┴─────┴────────┘
  → 타입이 전부 문자열                → df['salary'].mean() 가능!
```

### → AI 연결: Pandas → ML 파이프라인의 시작점

> 머신러닝 프로젝트의 **80%는 데이터 전처리**이다.
> 데이터를 불러오고, 정리하고, 변환하는 모든 과정이 Pandas로 이루어진다.
>
> ```
> CSV 파일 → [Pandas] → 전처리 → [Scikit-learn] → 모델 학습 → 예측
> ```

---

## 2.2 **데이터프레임**(DataFrame)과 **시리즈**(Series)

In [ ]:
# ── 시리즈(Series): 1차원 라벨 배열 ──
s = pd.Series([10, 20, 30, 40], index=['a', 'b', 'c', 'd'])
print(f"[시리즈]\n{s}")
print(f"[인덱싱] s['b'] = {s['b']}")
print(f"[타입] type(s) = {type(s)}")
print()

# ── 데이터프레임(DataFrame): 2차원 라벨 테이블 ──
df = pd.DataFrame({
    'name': ['Alice', 'Bob', 'Carol', 'Dave'],
    'age': [25, 30, 28, 35],
    'city': ['Seoul', 'Busan', 'Seoul', 'Daegu'],
    'salary': [50000, 60000, 55000, 70000]
})
print(f"[데이터프레임]\n{df}")
print(f"\n[열 선택] df['name'] =\n{df['name']}")    # → Series
print(f"\n[복수 열] df[['name','salary']] =\n{df[['name','salary']]}")  # → DataFrame
print(f"\n[행 선택] df.iloc[1] =\n{df.iloc[1]}")     # → Series (1번 행)
print(f"\n[특정 셀] df.loc[2, 'city'] = {df.loc[2, 'city']}")  # 'Seoul'

```
시각화: DataFrame = Series들의 모음

       name     age     city     salary
       ┌────┐  ┌────┐  ┌──────┐  ┌──────┐
    0  │Alice│  │ 25 │  │Seoul │  │50000 │
    1  │ Bob │  │ 30 │  │Busan │  │60000 │
    2  │Carol│  │ 28 │  │Seoul │  │55000 │
    3  │Dave │  │ 35 │  │Daegu │  │70000 │
       └────┘  └────┘  └──────┘  └──────┘
        ↑ Series  ↑ Series  ↑ Series  ↑ Series

  DataFrame = 같은 인덱스를 공유하는 Series들의 딕셔너리
```

---

## 2.3 CSV 파일 불러오기와 기본 탐색

In [ ]:
# ── 에어비앤비 데이터 불러오기 ──
# Colab에서 실행:
# !wget -q https://raw.githubusercontent.com/...  # URL 직접 로드
# 또는 파일 업로드 후:
df = pd.read_csv('AB_NYC_2019.csv')

# ── 기본 탐색 5종 세트 ──
print("=" * 60)
print("1. head() — 처음 5행 미리보기")
print("=" * 60)
print(df.head())
print()

print("=" * 60)
print("2. tail() — 마지막 5행 미리보기")
print("=" * 60)
print(df.tail())
print()

print("=" * 60)
print("3. shape — 행×열 크기")
print("=" * 60)
print(f"행: {df.shape[0]:,}개, 열: {df.shape[1]}개")
print()

print("=" * 60)
print("4. info() — 열별 타입, 결측값 현황")
print("=" * 60)
df.info()
print()

print("=" * 60)
print("5. describe() — 수치형 열의 기술 통계")
print("=" * 60)
print(df.describe())

```
시각화: 기본 탐색 5종 세트 요약

┌─────────────────────────────────────────────┐
│  탐색 메서드 5종 세트 (반드시 외울 것!)       │
├──────────────┬──────────────────────────────┤
│  df.head()   │ 처음 5행 미리보기             │
│  df.tail()   │ 마지막 5행 미리보기           │
│  df.shape    │ (행 수, 열 수) 튜플           │
│  df.info()   │ 열 이름, 타입, 결측값 수      │
│  df.describe()│ 수치형 열의 평균/표준편차/사분위수│
└──────────────┴──────────────────────────────┘

▶ 새 데이터를 받으면 무조건 이 5개를 먼저 실행하라!
```

---

## 2.4 데이터 **필터링**(Filtering)

### ▶ 비유: 카페 메뉴판에서 원하는 것만 고르기

> "아메리카노만 보여줘" = 단일 조건 필터링
> "아메리카노이면서 아이스인 것만" = 복합 조건 필터링

In [ ]:
# ── 단일 조건 필터링 ──
manhattan = df[df['neighbourhood_group'] == 'Manhattan']
print(f"[맨해튼만] {len(manhattan):,}개 (전체 {len(df):,}개 중)")
print(f"[비율] {len(manhattan)/len(df)*100:.1f}%")
print()

# ── 복합 조건 필터링 ──
cheap_manhattan = df[(df['neighbourhood_group'] == 'Manhattan') & 
                     (df['price'] <= 100)]
print(f"[맨해튼 + $100 이하] {len(cheap_manhattan):,}개")
print()

# ── 다중 값 필터링 (isin) ──
boroughs = ['Manhattan', 'Brooklyn']
mb = df[df['neighbourhood_group'].isin(boroughs)]
print(f"[맨해튼 or 브루클린] {len(mb):,}개")
print()

# ── 문자열 조건 ──
hotel = df[df['name'].str.contains('Hotel', na=False)]
print(f"[이름에 'Hotel' 포함] {len(hotel)}개")

```
시각화: 필터링 과정

전체 데이터: 48,895행
     ↓ df['neighbourhood_group'] == 'Manhattan'
맨해튼만: 21,661행  (44.3%)
     ↓ & (df['price'] <= 100)
맨해튼 $100 이하: 15,234행
     ↓
  이것이 분석 대상 데이터!
```

---

## 2.5 **정렬**(Sorting)과 **집계**(Aggregation)

In [ ]:
# ── 정렬 ──
sorted_by_price = df.sort_values('price', ascending=False)
print(f"[가장 비싼 숙소 Top 5]")
print(sorted_by_price[['name', 'neighbourhood_group', 'price']].head())
print()

# ── groupby: 그룹별 집계 ──
region_stats = df.groupby('neighbourhood_group')['price'].agg(['mean', 'median', 'count'])
print(f"[지역별 가격 통계]\n{region_stats}")
print()

# ── 여러 열에 대한 groupby ──
room_region = df.groupby(['neighbourhood_group', 'room_type'])['price'].mean()
print(f"[지역 × 숙소유형별 평균 가격]\n{room_region}")

```
시각화: groupby 동작 원리

원본 데이터:
  ┌──────────┬───────────┬───────┐
  │  region  │ room_type │ price │
  ├──────────┼───────────┼───────┤
  │Manhattan │  Entire   │  200  │
  │Brooklyn  │  Private  │   80  │
  │Manhattan │  Private  │  100  │
  │Brooklyn  │  Entire   │  150  │
  │Manhattan │  Entire   │  180  │
  └──────────┴───────────┴───────┘
       ↓ groupby('region')
  
  ┌────────────┐    ┌───────────┐
  │ Manhattan  │    │ Brooklyn  │
  ├────────────┤    ├───────────┤
  │ 200, 100,  │    │ 80, 150   │
  │ 180        │    │           │
  └────────────┘    └───────────┘
       ↓ .mean()         ↓ .mean()
     160.0              115.0
```

### → AI 연결: **특성 엔지니어링**(Feature Engineering)

> groupby는 ML에서 **새로운 특성을 만드는** 핵심 도구이다.
> 예: "이 숙소가 있는 지역의 평균 가격"을 새 열로 추가하면
> 모델이 지역별 가격 수준을 학습할 수 있다.
>
> ```python
> region_mean = df.groupby('neighbourhood_group')['price'].transform('mean')
> df['region_avg_price'] = region_mean
> ```

---

## 2.6 **결측값**(Missing Values) 처리

### ▶ 비유: 시험지에 빈칸이 있을 때

> 학생이 1번, 3번, 5번만 답을 쓰고 2번, 4번은 비워놨다.
> **선택 1:** 빈칸은 무시하고 3문제만 채점한다 (dropna)
> **선택 2:** 빈칸을 0점으로 채운다 (fillna(0))
> **선택 3:** 빈칸을 평균 점수로 채운다 (fillna(mean))

In [ ]:
# ── 결측값 확인 ──
print(f"[열별 결측값 수]")
print(df.isnull().sum())
print()
print(f"[열별 결측값 비율]")
print((df.isnull().sum() / len(df) * 100).round(1))
print()

# ── 결측값 처리 방법 ──
# 방법 1: 삭제
df_dropped = df.dropna(subset=['name'])
print(f"[dropna] 삭제 전: {len(df)} → 삭제 후: {len(df_dropped)}")
print()

# 방법 2: 채우기 (0으로)
df_filled_0 = df['reviews_per_month'].fillna(0)
print(f"[fillna(0)] 결측 → 0으로 채움")
print(f"  채우기 전 평균: {df['reviews_per_month'].mean():.2f}")
print(f"  채우기 후 평균: {df_filled_0.mean():.2f}")
print()

# 방법 3: 채우기 (중앙값으로)
median_val = df['reviews_per_month'].median()
df_filled_med = df['reviews_per_month'].fillna(median_val)
print(f"[fillna(median)] 결측 → 중앙값({median_val})으로 채움")
print(f"  채우기 후 평균: {df_filled_med.mean():.2f}")

```
시각화: 결측값 처리 전략 비교

원본:  [85, NaN, 92, NaN, 78, 90]
         ↓        ↓        ↓        ↓
dropna: [85,      92,      78, 90]  → 4개로 줄어듦 (정보 손실)
fillna(0): [85, 0, 92, 0, 78, 90]  → 평균이 크게 낮아짐
fillna(mean): [85, 86.25, 92, 86.25, 78, 90]  → 분포를 크게 왜곡하지 않음
fillna(median): [85, 87.5, 92, 87.5, 78, 90]  → 이상치에 강건

▶ 일반 규칙:
  - 결측 비율 < 5%:  dropna가 안전하다
  - 결측 비율 5~30%: fillna(median) 또는 fillna(mean)
  - 결측 비율 > 30%: 해당 열 자체를 삭제 고려
```

### → AI 연결: 결측값 처리가 모델 성능을 결정한다

> **Scikit-learn**의 모델들은 결측값(NaN)을 **입력으로 받지 못한다.**
> 결측값을 잘못 처리하면 → 모델이 잘못된 패턴을 학습한다.
> 6주차에서 배울 `SimpleImputer`는 이 과정을 자동화한다.

---

## 2.7 **인터랙티브 위젯**: Pandas 필터링 탐색기

In [ ]:
# ── 위젯 3: 지역별 가격 탐색기 ──
def region_explorer(region, max_price, room_type):
    filtered = df[(df['neighbourhood_group'] == region) & 
                  (df['price'] <= max_price)]
    if room_type != '전체':
        filtered = filtered[filtered['room_type'] == room_type]
    
    clear_output(wait=True)
    print(f"┌─────────────────────────────────────────────┐")
    print(f"│   에어비앤비 필터링 탐색기                  │")
    print(f"├─────────────────────────────────────────────┤")
    print(f"│  지역: {region:<15} 숙소유형: {room_type:<12}│")
    print(f"│  최대 가격: ${max_price:<10}                  │")
    print(f"├─────────────────────────────────────────────┤")
    print(f"│  검색 결과: {len(filtered):,}개               │")
    if len(filtered) > 0:
        print(f"│  평균 가격: ${filtered['price'].mean():.0f}   │")
        print(f"│  중앙 가격: ${filtered['price'].median():.0f} │")
        print(f"│  최저: ${filtered['price'].min()} ~ 최고: ${filtered['price'].max()} │")
    print(f"└─────────────────────────────────────────────┘")

region_dd = widgets.Dropdown(options=df['neighbourhood_group'].unique(),
                             description='지역:')
price_sl = widgets.IntSlider(value=200, min=10, max=1000, step=10,
                             description='최대 가격:')
room_dd = widgets.Dropdown(options=['전체'] + list(df['room_type'].unique()),
                           description='숙소 유형:')
widgets.interact(region_explorer, region=region_dd, 
                 max_price=price_sl, room_type=room_dd)

---

## 2.8 Pandas **연습문제** (기초)

### 문제 6. DataFrame 생성과 기본 탐색

In [ ]:
# 다음 데이터로 DataFrame을 만들고, info()와 describe()를 실행하라.
# 학생 이름: ['김철수', '이영희', '박민수', '최지은', '정하늘']
# 수학 점수: [85, 92, 78, 95, 88]
# 영어 점수: [90, 87, 82, 91, 79]
# 과학 점수: [88, 95, 75, 89, 93]

# 여기에 코드를 작성하라

### 문제 7. 필터링과 조건부 집계

In [ ]:
# 에어비앤비 데이터에서:
# (1) Brooklyn 지역의 'Private room'만 필터링하라.
# (2) 해당 데이터의 평균 가격, 중앙값 가격, 행 수를 출력하라.

# 여기에 코드를 작성하라

### 문제 8. groupby 활용

In [ ]:
# 에어비앤비 데이터에서:
# (1) room_type별 평균 가격을 구하라.
# (2) neighbourhood_group별 결측값 비율을 구하라. (reviews_per_month 열)

# 여기에 코드를 작성하라

### 문제 9. 결측값 처리 비교

In [ ]:
# reviews_per_month 열에 대해:
# (1) dropna했을 때의 평균
# (2) fillna(0)했을 때의 평균
# (3) fillna(median)했을 때의 평균
# 세 가지를 비교하고, 어떤 방법이 가장 적절한지 이유와 함께 서술하라.

# 여기에 코드를 작성하라

### 문제 10. 종합 분석

In [ ]:
# 에어비앤비 데이터에서 '가격이 $50~$200인 Manhattan의 Entire home/apt'만 추출하고,
# neighbourhood별 평균 가격 Top 5를 구하라.

# 여기에 코드를 작성하라

<details>
<summary>▶ Pandas 기초 연습문제 해답 (클릭하여 펼치기)</summary>

#### 해답 6

<pre><code class="python">df_students = pd.DataFrame({
    'name': ['김철수', '이영희', '박민수', '최지은', '정하늘'],
    'math': [85, 92, 78, 95, 88],
    'english': [90, 87, 82, 91, 79],
    'science': [88, 95, 75, 89, 93]
})
print(df_students)
print()
df_students.info()
print()
print(df_students.describe())</code></pre>

#### 해답 7

<pre><code class="python">brooklyn_private = df[(df['neighbourhood_group'] == 'Brooklyn') &amp; 
                      (df['room_type'] == 'Private room')]
print(f"[행 수] {len(brooklyn_private):,}개")
print(f"[평균 가격] ${brooklyn_private['price'].mean():.1f}")
print(f"[중앙값]   ${brooklyn_private['price'].median():.1f}")</code></pre>

#### 해답 8

<pre><code class="python"># (1)
print(df.groupby('room_type')['price'].mean())
print()

# (2)
missing_rate = df.groupby('neighbourhood_group')['reviews_per_month'].apply(
    lambda x: x.isnull().mean() * 100
)
print(f"[지역별 reviews_per_month 결측 비율]\n{missing_rate.round(1)}")</code></pre>

#### 해답 9

<pre><code class="python">original_mean = df['reviews_per_month'].mean()
dropna_mean = df['reviews_per_month'].dropna().mean()
fillna0_mean = df['reviews_per_month'].fillna(0).mean()
fillna_med_mean = df['reviews_per_month'].fillna(df['reviews_per_month'].median()).mean()

print(f"[원본 평균 (NaN 제외)] {dropna_mean:.2f}")
print(f"[fillna(0)]            {fillna0_mean:.2f}")
print(f"[fillna(median)]       {fillna_med_mean:.2f}")
print()
print("[분석] fillna(0)은 평균을 크게 낮추므로 부적절하다.")
print("       fillna(median)이 원본 분포를 가장 잘 보존한다.")
print("       dropna는 10,052개 행이 사라지므로 정보 손실이 크다.")</code></pre>

#### 해답 10

<pre><code class="python">filtered = df[(df['price'] &gt;= 50) &amp; (df['price'] &lt;= 200) &amp; 
              (df['neighbourhood_group'] == 'Manhattan') &amp;
              (df['room_type'] == 'Entire home/apt')]
top5 = filtered.groupby('neighbourhood')['price'].mean().sort_values(ascending=False).head()
print(f"[Manhattan Entire home $50~$200 동네별 평균가격 Top 5]\n{top5}")</code></pre>

</details>

---

# PART 3: **프로젝트 실습** — "뉴욕 에어비앤비 적정 숙박료 분석"

---

## 미션

> **"뉴욕에서 방이 둘 딸린 집을 에어비앤비에 내놓으려 한다. 적정 숙박료는 얼마인가?"**

### 분석 파이프라인 7단계

```
단계 1: 데이터 불러오기 + 구조 파악
   ↓
단계 2: 2베드룸 필터링
   ↓
단계 3: 결측값 처리
   ↓
단계 4: 지역별(neighbourhood_group) 가격 통계
   ↓
단계 5: 이상치 제거 전후 비교
   ↓
단계 6: 검증 — mean vs median, 필터링 전후 행 수
   ↓
단계 7: 최종 결론 — "적정 숙박료는 $___이다"
```

---

### 단계 1: 데이터 불러오기 + 구조 파악

In [ ]:
import pandas as pd
import numpy as np

# ── 데이터 불러오기 ──
df = pd.read_csv('AB_NYC_2019.csv')

# ── 구조 파악 ──
print(f"{'='*60}")
print(f"  뉴욕 에어비앤비 데이터셋 구조 파악")
print(f"{'='*60}")
print(f"[행 × 열] {df.shape[0]:,}행 × {df.shape[1]}열")
print()

print(f"[열 목록]")
for i, col in enumerate(df.columns):
    dtype = df[col].dtype
    null_count = df[col].isnull().sum()
    null_pct = null_count / len(df) * 100
    print(f"  {i:2d}. {col:<25} {str(dtype):<10} 결측: {null_count:,}개 ({null_pct:.1f}%)")
print()

print(f"[수치형 통계]")
print(df.describe().round(1))
print()

print(f"[범주형 값 확인]")
print(f"  neighbourhood_group: {df['neighbourhood_group'].unique()}")
print(f"  room_type: {df['room_type'].unique()}")

---

### 단계 2: 2베드룸 필터링

In [ ]:
# ⚠️ 주의: AB_NYC_2019 데이터셋에는 'bedrooms' 열이 없다!
# 'calculated_host_listings_count'와 'room_type'을 기준으로 분석한다.
# 'Entire home/apt'를 선택하는 것이 "방이 둘 딸린 집"에 가장 가깝다.

df_entire = df[df['room_type'] == 'Entire home/apt']
print(f"[필터링] Entire home/apt만 선택")
print(f"  전체: {len(df):,}행 → 필터링 후: {len(df_entire):,}행")
print(f"  비율: {len(df_entire)/len(df)*100:.1f}%")

---

### 단계 3: 결측값 처리

In [ ]:
print(f"[결측값 현황 — Entire home/apt만]")
null_info = df_entire.isnull().sum()
null_info = null_info[null_info > 0]
for col, count in null_info.items():
    pct = count / len(df_entire) * 100
    print(f"  {col}: {count:,}개 ({pct:.1f}%)")

# ── 처리 전략 ──
df_clean = df_entire.copy()

# name: 결측이 적으므로 삭제
df_clean = df_clean.dropna(subset=['name'])

# reviews_per_month: 리뷰가 없는 숙소 → 0으로 채움
df_clean['reviews_per_month'] = df_clean['reviews_per_month'].fillna(0)

# last_review: NaT(날짜 결측) → 분석에 미사용
print(f"\n[처리 후] {len(df_clean):,}행, 결측값 수: {df_clean[['name','price','reviews_per_month']].isnull().sum().sum()}")

---

### 단계 4: 지역별 가격 통계

In [ ]:
print(f"{'='*70}")
print(f"  지역별 가격 통계 (Entire home/apt)")
print(f"{'='*70}")
region_stats = df_clean.groupby('neighbourhood_group')['price'].agg(
    ['count', 'mean', 'median', 'std', 'min', 'max']
).round(1)
region_stats.columns = ['숙소 수', '평균($)', '중앙값($)', '표준편차', '최소($)', '최대($)']
print(region_stats)
print()

# 비율 확인
print(f"[지역별 숙소 비율]")
for region in region_stats.index:
    count = region_stats.loc[region, '숙소 수']
    pct = count / region_stats['숙소 수'].sum() * 100
    bar = "█" * int(pct)
    print(f"  {region:<15} {count:5.0f}개 ({pct:4.1f}%) {bar}")

---

### 단계 5: 이상치 제거 전후 비교

In [ ]:
# ── 이상치 기준: 가격 > $1,000 ──
print(f"[이상치 분석]")
print(f"  $1,000 초과 숙소: {(df_clean['price'] > 1000).sum()}개")
print(f"  $5,000 초과 숙소: {(df_clean['price'] > 5000).sum()}개")
print(f"  $10,000 숙소: {(df_clean['price'] == 10000).sum()}개 ← 최댓값")
print()

# ── 이상치 제거 ──
df_no_outlier = df_clean[df_clean['price'] <= 1000]

# ── 전후 비교 ──
print(f"{'='*50}")
print(f"  이상치 제거 전후 비교")
print(f"{'='*50}")
print(f"{'항목':<12} {'제거 전':>12} {'제거 후':>12}")
print(f"{'-'*50}")
print(f"{'행 수':<12} {len(df_clean):>12,} {len(df_no_outlier):>12,}")
print(f"{'평균 가격':<12} ${df_clean['price'].mean():>11.1f} ${df_no_outlier['price'].mean():>11.1f}")
print(f"{'중앙값':<12} ${df_clean['price'].median():>11.1f} ${df_no_outlier['price'].median():>11.1f}")
print(f"{'표준편차':<12} ${df_clean['price'].std():>11.1f} ${df_no_outlier['price'].std():>11.1f}")
print()
print(f"[관찰] 이상치 제거 후 평균이 크게 낮아지지만, 중앙값은 크게 변하지 않는다.")
print(f"       → 평균은 이상치에 민감하고, 중앙값은 강건하다.")

---

### 단계 6: 검증

In [ ]:
# ── 최종 검증 ──
print(f"{'='*60}")
print(f"  최종 검증 — 이상치 제거 후 지역별 통계")
print(f"{'='*60}")
final_stats = df_no_outlier.groupby('neighbourhood_group')['price'].agg(
    ['count', 'mean', 'median']
).round(1)
final_stats.columns = ['숙소 수', '평균($)', '중앙값($)']
print(final_stats)
print()

# mean vs median 차이 분석
for region in final_stats.index:
    mean_val = final_stats.loc[region, '평균($)']
    median_val = final_stats.loc[region, '중앙값($)']
    diff_pct = (mean_val - median_val) / median_val * 100
    skew = "오른쪽 꼬리 (비싼 숙소 존재)" if diff_pct > 10 else "대칭에 가까움"
    print(f"  {region}: 평균 ${mean_val} vs 중앙값 ${median_val} (차이 {diff_pct:.1f}%) → {skew}")

---

### 단계 7: 최종 결론

In [ ]:
# ── 적정 숙박료 제안 ──
print(f"{'='*60}")
print(f"   최종 결론: 적정 숙박료 제안")
print(f"{'='*60}")
print()
print(f"  [질문] 뉴욕에서 Entire home/apt를 내놓으려 한다.")
print(f"         적정 숙박료는 얼마인가?")
print()
print(f"  [답변]")
for region in final_stats.index:
    median_price = final_stats.loc[region, '중앙값($)']
    print(f"    {region:<15} → ${median_price:.0f}/박 (중앙값 기준)")
print()
print(f"  [권장] 중앙값을 기준으로 설정하되,")
print(f"         시설·위치·리뷰 수에 따라 ±20% 조정하라.")
print(f"  [근거] 평균은 고가 숙소에 의해 왜곡되므로,")
print(f"         중앙값이 더 현실적인 지표이다.")

---

# PART 4: **신경망 기초**(Neural Network Fundamentals) — 심화

---

## 4.1 **퍼셉트론**(Perceptron) — 신경망의 최소 단위

### ▶ 비유: 의사결정 과정

> 점심 메뉴를 고를 때, 여러 조건을 **종합**해서 결정한다:
> - 맛(weight=0.5): 맛있으면 +1, 아니면 0
> - 가격(weight=0.3): 싸면 +1, 아니면 0
> - 거리(weight=0.2): 가까우면 +1, 아니면 0
>
> **종합 점수** = 맛×0.5 + 가격×0.3 + 거리×0.2
> **결정** = 종합 점수가 **임계값**(Threshold)을 넘으면 "간다!", 아니면 "안 간다"
>
> 이것이 바로 **퍼셉트론**이다!

In [ ]:
# ── 퍼셉트론 구현 ──
def perceptron(inputs, weights, bias):
    """단일 퍼셉트론 계산"""
    z = np.dot(inputs, weights) + bias       # 가중합
    output = 1 if z >= 0 else 0              # 계단 함수
    return z, output

# ── AND 게이트 ──
print(f"{'='*50}")
print(f"  AND 게이트 — 퍼셉트론으로 구현")
print(f"{'='*50}")
weights = np.array([0.5, 0.5])
bias = -0.7

for x1, x2 in [(0,0), (0,1), (1,0), (1,1)]:
    inputs = np.array([x1, x2])
    z, out = perceptron(inputs, weights, bias)
    print(f"  입력: ({x1}, {x2}) → 가중합: {z:.1f} → 출력: {out}  {'✅' if out == (x1 and x2) else '❌'}")

print()

# ── OR 게이트 ──
print(f"{'='*50}")
print(f"  OR 게이트 — 퍼셉트론으로 구현")
print(f"{'='*50}")
weights_or = np.array([0.5, 0.5])
bias_or = -0.2

for x1, x2 in [(0,0), (0,1), (1,0), (1,1)]:
    inputs = np.array([x1, x2])
    z, out = perceptron(inputs, weights_or, bias_or)
    expected = 1 if (x1 or x2) else 0
    print(f"  입력: ({x1}, {x2}) → 가중합: {z:.1f} → 출력: {out}  {'✅' if out == expected else '❌'}")

```
시각화: 퍼셉트론 구조

   입력           가중치          가중합           활성화 함수        출력
  ┌────┐
  │ x₁ │──── w₁ ───┐
  └────┘            │
                    ├──→ Σ (x₁w₁ + x₂w₂ + b) ──→ f(z) ──→ y
  ┌────┐            │
  │ x₂ │──── w₂ ───┘
  └────┘       ↑
              ┌──┐
              │ b │  ← 편향(Bias)
              └──┘

  AND 게이트:                        OR 게이트:
  w=[0.5, 0.5], b=-0.7              w=[0.5, 0.5], b=-0.2
  (0,0): 0+0-0.7 = -0.7 → 0 ✅     (0,0): 0+0-0.2 = -0.2 → 0 ✅
  (0,1): 0+0.5-0.7 = -0.2 → 0 ✅   (0,1): 0+0.5-0.2 = 0.3 → 1 ✅
  (1,0): 0.5+0-0.7 = -0.2 → 0 ✅   (1,0): 0.5+0-0.2 = 0.3 → 1 ✅
  (1,1): 0.5+0.5-0.7 = 0.3 → 1 ✅  (1,1): 0.5+0.5-0.2 = 0.8 → 1 ✅
```

### XOR 문제 — 퍼셉트론의 한계

In [ ]:
# ── XOR 게이트: 단일 퍼셉트론으로는 불가능! ──
print(f"{'='*50}")
print(f"  XOR 게이트 — 단일 퍼셉트론의 한계")
print(f"{'='*50}")
print(f"  XOR 진리표:")
print(f"  (0,0) → 0")
print(f"  (0,1) → 1")
print(f"  (1,0) → 1")
print(f"  (1,1) → 0  ← 둘 다 1인데 0이 나와야 한다!")
print()
print(f"  [문제] 직선 하나로는 이 4개의 점을 분리할 수 없다!")
print(f"  [해결] 퍼셉트론을 여러 개 쌓으면 된다 → 다층 퍼셉트론(MLP)")

```
시각화: XOR — 직선으로 분리 불가능

     x₂
     1 │  ● (0,1)=1      ○ (1,1)=0
       │
       │
     0 │  ○ (0,0)=0      ● (1,0)=1
       └──────────────────────── x₁
       0                  1

  ● = 출력 1,  ○ = 출력 0
  → 직선 하나로 ●와 ○를 분리할 수 없다!
  → 해결: 층을 쌓아서 비선형 경계를 만든다!
```

---

## 4.2 **활성화 함수**(Activation Functions) 6종

### ▶ 비유: 볼륨 조절 방식

> **계단 함수** = 스위치 (ON/OFF만 가능)
> **시그모이드** = 부드러운 디머 (0~1 사이에서 서서히 변화)
> **ReLU** = 한쪽만 열린 문 (음수는 막고, 양수는 그대로 통과)
> **Softmax** = 파이 차트 (합이 1이 되도록 비율을 만듦)

In [ ]:
import numpy as np

# ── 활성화 함수 정의 ──
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def relu(z):
    return np.maximum(0, z)

def leaky_relu(z, alpha=0.01):
    return np.where(z > 0, z, alpha * z)

def tanh_func(z):
    return np.tanh(z)

def gelu(z):
    """GELU — GPT, BERT에서 사용하는 최신 활성화 함수"""
    return 0.5 * z * (1 + np.tanh(np.sqrt(2/np.pi) * (z + 0.044715 * z**3)))

def softmax(z):
    exp_z = np.exp(z - np.max(z))  # 오버플로 방지
    return exp_z / exp_z.sum()

# ── 값 범위 생성 ──
z = np.linspace(-5, 5, 100)

# ── 각 함수의 출력 ──
print(f"{'='*60}")
print(f"  활성화 함수 6종 비교 (z = -5 ~ 5)")
print(f"{'='*60}")
print(f"{'함수':<12} {'범위':<20} {'특징'}")
print(f"{'-'*60}")
print(f"{'Sigmoid':<12} {'(0, 1)':<20} 이진 분류 출력층에 사용")
print(f"{'Tanh':<12} {'(-1, 1)':<20} 은닉층, 0 중심")
print(f"{'ReLU':<12} {'[0, ∞)':<20} 가장 많이 사용, 빠름")
print(f"{'Leaky ReLU':<12} {'(-∞, ∞)':<20} 죽은 뉴런 문제 해결")
print(f"{'GELU':<12} {'(-0.17, ∞)':<20} GPT/BERT에서 사용")
print(f"{'Softmax':<12} {'(0, 1), 합=1':<20} 다중 분류 출력층")
print()

# ── 특정 값에서의 출력 비교 ──
test_values = [-2, -1, 0, 1, 2]
print(f"{'z':>4} {'Sigmoid':>10} {'Tanh':>10} {'ReLU':>10} {'LeakyReLU':>10} {'GELU':>10}")
print(f"{'-'*54}")
for v in test_values:
    print(f"{v:4d} {sigmoid(v):10.4f} {tanh_func(v):10.4f} {relu(v):10.4f} {leaky_relu(v):10.4f} {gelu(v):10.4f}")

```
시각화: 활성화 함수 그래프 (ASCII 버전)

Sigmoid:                     ReLU:
 1.0 ┤        ╭──────        ┤         ╱
     │       ╱                │        ╱
 0.5 ┤──────╱                 │       ╱
     │     ╱                  │      ╱
 0.0 ┤────╯                   ┤─────╱
     └────────────────        └────────────────
    -5    0    5             -5    0    5

Tanh:                        GELU (GPT용):
 1.0 ┤        ╭──────        ┤         ╱
     │       ╱                │        ╱
 0.0 ┤──────╱                 │  ╭───╱
     │     ╱                  │ ╯  ╱
-1.0 ┤────╯                   ┤───╯
     └────────────────        └────────────────
    -5    0    5             -5    0    5

Leaky ReLU:                  Softmax (예시):
     ┤         ╱              입력: [2.0, 1.0, 0.1]
     │        ╱                      ↓
     │       ╱               출력: [0.659, 0.242, 0.099]
     │      ╱                      ↓ 합 = 1.000
     ┤─────╱                 "고양이 66%, 개 24%, 새 10%"
     └──╱──────────
    -5    0    5
```

### → AI 연결: 어떤 활성화 함수를 언제 사용하는가?

> | 위치 | 추천 활성화 함수 | 이유 |
> |---|---|---|
> | **은닉층**(Hidden Layer) | **ReLU** 또는 **GELU** | 빠르고 기울기 소실 문제 적음 |
> | **이진 분류 출력층** | **시그모이드**(Sigmoid) | 확률(0~1)을 출력 |
> | **다중 분류 출력층** | **소프트맥스**(Softmax) | 클래스별 확률 (합=1) |
> | **회귀 출력층** | 없음(항등 함수) | 실수 값 그대로 출력 |
> | **GPT, BERT** | **GELU** | 확률적 마스킹 효과 |

---

## 4.3 **손실 함수**(Loss Function) 시각화

### ▶ 비유: 양궁 과녁에서 빗나간 거리

> 모델의 예측이 과녁의 중심(정답)에서 얼마나 떨어져 있는지를 **숫자로 측정**한 것이
> **손실 함수**(Loss Function)이다.
> - 과녁 중심에 가까울수록 → 손실이 작다 (좋은 모델)
> - 과녁에서 멀수록 → 손실이 크다 (나쁜 모델)

In [ ]:
# ── MSE (Mean Squared Error) — 회귀용 ──
def mse_loss(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

y_true = np.array([3.0, 5.0, 7.0, 9.0])
predictions = np.linspace(0, 12, 50)

print(f"{'='*60}")
print(f"  MSE 손실 함수 — 예측값에 따른 손실 변화")
print(f"{'='*60}")
print(f"  정답: {y_true}")
print()

# 예측 1: 정확한 예측
pred1 = np.array([3.0, 5.0, 7.0, 9.0])
print(f"  [정확한 예측] {pred1} → MSE = {mse_loss(y_true, pred1):.4f}")

# 예측 2: 약간 빗나감
pred2 = np.array([3.5, 4.5, 7.5, 8.5])
print(f"  [약간 빗나감] {pred2} → MSE = {mse_loss(y_true, pred2):.4f}")

# 예측 3: 크게 빗나감
pred3 = np.array([1.0, 2.0, 3.0, 4.0])
print(f"  [크게 빗나감] {pred3} → MSE = {mse_loss(y_true, pred3):.4f}")
print()

# ── Cross-Entropy Loss — 분류용 ──
def cross_entropy_loss(y_true, y_pred):
    """이진 교차 엔트로피"""
    epsilon = 1e-15  # log(0) 방지
    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

print(f"{'='*60}")
print(f"  Cross-Entropy 손실 함수 — 분류 문제")
print(f"{'='*60}")
y_true_cls = np.array([1, 0, 1, 1, 0])  # 정답 라벨

# 좋은 예측
pred_good = np.array([0.9, 0.1, 0.8, 0.95, 0.05])
print(f"  [좋은 예측] {pred_good}")
print(f"    CE Loss = {cross_entropy_loss(y_true_cls, pred_good):.4f}")

# 나쁜 예측
pred_bad = np.array([0.3, 0.7, 0.4, 0.5, 0.8])
print(f"  [나쁜 예측] {pred_bad}")
print(f"    CE Loss = {cross_entropy_loss(y_true_cls, pred_bad):.4f}")

# 최악의 예측 (정반대)
pred_worst = np.array([0.1, 0.9, 0.1, 0.1, 0.9])
print(f"  [최악 예측] {pred_worst}")
print(f"    CE Loss = {cross_entropy_loss(y_true_cls, pred_worst):.4f}")

```
시각화: MSE와 Cross-Entropy 비교

MSE (회귀):                     Cross-Entropy (분류):
  손실                             손실
   │                                │
 25│\                            3.0│\
   │ \                              │ \
 16│  \                          2.0│  \
   │   \                            │   \
  4│    \    ╱                   1.0│    \
   │     \  ╱                       │     \
  0│      ╰╯                    0.0│      ╰──
   └───────────                    └───────────
   0   정답  10                   0   정답   1
     (예측값)                      (예측 확률)

  MSE: 오차²를 평균                CE: -log(정답 확률)
  → 큰 오차에 강하게 벌점         → 틀리면 ∞로 폭발!
```

### → AI 연결: 손실 함수 = 학습의 나침반

> 모든 신경망 학습은 이 두 단계의 반복이다:
> 1. **순전파**: 예측값 계산 → 손실 함수로 "얼마나 틀렸는지" 측정
> 2. **역전파**: 손실을 줄이는 방향으로 가중치 조정
>
> ```
> 입력 → [신경망] → 예측 → [손실함수] → 손실값 → [역전파] → 가중치 업데이트
>   ↑_______________________________________________________↓  ← 이걸 수만 번 반복!
> ```

---

## 4.4 **경사 하강법**(Gradient Descent) 직관

### ▶ 비유: 안개 낀 산에서 내려오기

> 안개가 자욱한 산꼭대기에 서 있다. 아래로 내려가야 한다.
> 눈이 보이지 않으니, **발밑의 경사(기울기)**만 느낀다.
> → **가장 가파른 내리막 방향으로 한 걸음씩** 내려간다.
> → 이것을 반복하면 결국 **골짜기(최솟값)**에 도달한다!
>
> - **경사**(Gradient) = 현재 위치에서의 기울기 = 어느 방향으로 가야 손실이 줄어드는가
> - **학습률**(Learning Rate) = 한 걸음의 크기 = 너무 크면 지나치고, 너무 작으면 느리다

In [ ]:
# ── 경사 하강법 시뮬레이션: y = (x-3)² + 1 의 최솟값 찾기 ──

def loss_fn(x):
    """손실 함수: (x-3)² + 1"""
    return (x - 3) ** 2 + 1

def gradient(x):
    """손실 함수의 기울기: 2(x-3)"""
    return 2 * (x - 3)

# ── 학습률 비교 ──
learning_rates = [0.01, 0.1, 0.5]

for lr in learning_rates:
    x = 0.0  # 시작점
    print(f"\n{'='*50}")
    print(f"  학습률(lr) = {lr}")
    print(f"{'='*50}")
    print(f"  {'step':>4}  {'x':>8}  {'loss':>8}  {'gradient':>10}")
    print(f"  {'-'*36}")
    
    for step in range(10):
        loss = loss_fn(x)
        grad = gradient(x)
        print(f"  {step:4d}  {x:8.4f}  {loss:8.4f}  {grad:10.4f}")
        x = x - lr * grad  # 가중치 업데이트!
    
    print(f"  {'최종':>4}  {x:8.4f}  {loss_fn(x):8.4f}")
    print(f"  [정답: x=3.0, loss=1.0]")

```
시각화: 학습률에 따른 경사 하강법 경로

  손실
   │
 10│ ●                           lr=0.01: 느린 수렴
   │  ●                                ●→●→●→●→●→...→●→●
   │   ●
   │    ●                        lr=0.1: 적절한 수렴
   │     ●  ●                          ●→→→●→●→●
   │      ●   ●                              ↓
  1│───────────●── ← 최솟값       lr=0.5: 빠르지만 위험
   └────────────────── x               ●→→→→→→→●
   0          3                        (지나칠 수 있음!)

  ▶ 학습률이 너무 크면: 최솟값을 지나쳐서 발산할 수 있다
     학습률이 너무 작으면: 수렴이 너무 느리다
     적절한 학습률: 빠르게 수렴하면서도 안정적
```

---

## 4.5 **순전파**(Forward Propagation) — 1층 신경망

In [ ]:
# ── 1층 신경망: 입력 → 은닉층 → 출력 ──
np.random.seed(42)

# 데이터: 학생 3명, 특성 4개
X = np.array([[0.8, 0.2, 0.5, 0.9],   # 학생 1
              [0.3, 0.7, 0.1, 0.6],   # 학생 2
              [0.9, 0.4, 0.8, 0.3]])  # 학생 3

# 가중치와 편향 (랜덤 초기화)
W1 = np.random.randn(4, 3) * 0.5    # (입력 4) → (은닉 3)
b1 = np.zeros(3)
W2 = np.random.randn(3, 2) * 0.5    # (은닉 3) → (출력 2)
b2 = np.zeros(2)

print(f"{'='*60}")
print(f"  순전파 과정 — 단계별 추적")
print(f"{'='*60}")

# 단계 1: 입력 → 은닉층
print(f"\n[단계 1] 가중합: Z1 = X @ W1 + b1")
Z1 = X @ W1 + b1
print(f"  X shape:  {X.shape}")
print(f"  W1 shape: {W1.shape}")
print(f"  Z1 shape: {Z1.shape}")
print(f"  Z1 =\n{np.round(Z1, 4)}")

# 단계 2: 활성화 (ReLU)
print(f"\n[단계 2] 활성화: A1 = ReLU(Z1)")
A1 = relu(Z1)
print(f"  A1 =\n{np.round(A1, 4)}")
print(f"  (음수는 0으로 변환됨)")

# 단계 3: 은닉층 → 출력층
print(f"\n[단계 3] 가중합: Z2 = A1 @ W2 + b2")
Z2 = A1 @ W2 + b2
print(f"  A1 shape: {A1.shape}")
print(f"  W2 shape: {W2.shape}")
print(f"  Z2 shape: {Z2.shape}")
print(f"  Z2 =\n{np.round(Z2, 4)}")

# 단계 4: 활성화 (Sigmoid → 확률 출력)
print(f"\n[단계 4] 활성화: Y = Sigmoid(Z2)")
Y = sigmoid(Z2)
print(f"  Y (예측 확률) =\n{np.round(Y, 4)}")
print(f"  학생 1: 클래스 0 확률 {Y[0,0]:.2%}, 클래스 1 확률 {Y[0,1]:.2%}")
print(f"  학생 2: 클래스 0 확률 {Y[1,0]:.2%}, 클래스 1 확률 {Y[1,1]:.2%}")
print(f"  학생 3: 클래스 0 확률 {Y[2,0]:.2%}, 클래스 1 확률 {Y[2,1]:.2%}")

```
시각화: 1층 신경망 순전파 구조

  입력층 (4)       은닉층 (3)       출력층 (2)

  ┌────┐           ┌────┐           ┌────┐
  │ x₁ │────┐  ┌──→│ h₁ │────┐  ┌──→│ y₁ │  ← 클래스 0 확률
  └────┘    │  │   └────┘    │  │   └────┘
  ┌────┐    │  │   ┌────┐    │  │   ┌────┐
  │ x₂ │────┼──┤──→│ h₂ │────┼──┤──→│ y₂ │  ← 클래스 1 확률
  └────┘    │  │   └────┘    │  │   └────┘
  ┌────┐    │  │   ┌────┐    │  │
  │ x₃ │────┼──┘──→│ h₃ │────┘──┘
  └────┘    │      └────┘
  ┌────┐    │
  │ x₄ │────┘     Z1=X@W1+b1     Z2=A1@W2+b2
  └────┘          A1=ReLU(Z1)     Y=Sigmoid(Z2)

  [X (3×4)] → [W1 (4×3)] → [Z1 (3×3)] → [ReLU] → [A1 (3×3)]
            → [W2 (3×2)] → [Z2 (3×2)] → [Sigmoid] → [Y (3×2)]
```

### → AI 연결: 이것이 **딥러닝**의 전부이다

> 위 코드에서 한 것:
> 1. 가중합 = **행렬 곱셈** (NumPy의 @)
> 2. 활성화 = **비선형 변환** (ReLU, Sigmoid)
>
> 이 두 단계를 여러 번 쌓으면 = **딥러닝**
> 10층이면 = 10번 반복
> GPT는 96층 = 96번 반복 (+ 어텐션)
>
> **핵심:** 모든 딥러닝은 결국 NumPy 연산의 확장이다.

---

## 4.6 **역전파**(Backpropagation) 직관

### ▶ 비유: 요리에서 간 조절

> 짠 국을 끓였다고 하자. 어디서 소금이 많이 들어갔을까?
> 1. 마지막에 넣은 소금? → 출력층에서의 오차
> 2. 중간에 넣은 간장? → 은닉층의 기여도
> 3. 처음에 넣은 된장? → 입력층 가까이의 기여도
>
> **역전파** = 출력의 오차를 **거꾸로** 추적하여,
> 각 층의 가중치가 오차에 얼마나 기여했는지를 계산하는 것이다.

In [ ]:
# ── 역전파 직관: 간단한 계산 그래프 ──
print(f"{'='*60}")
print(f"  역전파 직관 — y = (x*w + b)² 에서 w를 업데이트")
print(f"{'='*60}")

x = 2.0    # 입력
w = 0.5    # 가중치 (이것을 학습!)
b = 1.0    # 편향
y_true = 7.0  # 정답

# 순전파
z = x * w + b          # z = 2*0.5 + 1 = 2
y_pred = z             # 예측값
loss = (y_pred - y_true) ** 2  # 손실 = (2-7)² = 25

print(f"\n[순전파]")
print(f"  z = x*w + b = {x}*{w} + {b} = {z}")
print(f"  loss = (y_pred - y_true)² = ({y_pred} - {y_true})² = {loss}")

# 역전파 (체인 룰)
dloss_dy = 2 * (y_pred - y_true)   # ∂loss/∂y_pred = 2(y-y_true)
dy_dw = x                          # ∂y/∂w = x
dloss_dw = dloss_dy * dy_dw        # 체인 룰!

print(f"\n[역전파 — 체인 룰(Chain Rule)]")
print(f"  ∂loss/∂y = 2(y_pred - y_true) = 2({y_pred}-{y_true}) = {dloss_dy}")
print(f"  ∂y/∂w = x = {dy_dw}")
print(f"  ∂loss/∂w = ∂loss/∂y × ∂y/∂w = {dloss_dy} × {dy_dw} = {dloss_dw}")

# 가중치 업데이트
lr = 0.01
w_new = w - lr * dloss_dw
print(f"\n[업데이트]")
print(f"  w_new = w - lr × ∂loss/∂w")
print(f"        = {w} - {lr} × {dloss_dw}")
print(f"        = {w_new}")
print(f"  새 예측: x*w_new + b = {x}*{w_new} + {b} = {x*w_new + b}")
print(f"  새 손실: ({x*w_new + b} - {y_true})² = {(x*w_new + b - y_true)**2:.4f}")
print(f"  [개선!] 손실이 {loss} → {(x*w_new + b - y_true)**2:.4f}로 줄어들었다.")

```
시각화: 역전파의 체인 룰(Chain Rule)

순전파 (→):
  x=2 ─── ×w=0.5 ──→ z=2 ──→ loss=(z-7)²=25
  b=1 ─── +b ──────↗

역전파 (←):
  ∂loss/∂w ← ∂loss/∂z × ∂z/∂w
            =  2(z-7)  ×   x
            = 2(2-7)   ×   2
            =   -10    ×   2
            =   -20

  w_new = 0.5 - 0.01 × (-20) = 0.5 + 0.2 = 0.7 ← 정답(3) 방향으로 이동!
```

---

## 4.7 **인터랙티브 위젯**: 경사 하강법 시뮬레이터

In [ ]:
# ── 위젯 4: 학습률 탐색기 ──
def gradient_descent_simulator(lr, steps):
    x = 0.0
    trajectory = [x]
    losses = [loss_fn(x)]
    
    for _ in range(steps):
        grad = gradient(x)
        x = x - lr * grad
        trajectory.append(x)
        losses.append(loss_fn(x))
    
    clear_output(wait=True)
    print(f"┌──────────────────────────────────────────────┐")
    print(f"│  경사 하강법 시뮬레이터                       │")
    print(f"│  목표: f(x) = (x-3)² + 1 의 최솟값 찾기      │")
    print(f"├──────────────────────────────────────────────┤")
    print(f"│  학습률: {lr}  |  스텝 수: {steps}            │")
    print(f"│  시작: x={trajectory[0]:.2f}  →  최종: x={trajectory[-1]:.4f} │")
    print(f"│  최종 손실: {losses[-1]:.6f}  (목표: 1.0)     │")
    print(f"├──────────────────────────────────────────────┤")
    
    # 경로 시각화
    max_loss = max(losses)
    for i in range(min(len(losses), 15)):
        bar_len = int(losses[i] / max_loss * 30) if max_loss > 0 else 0
        bar = "█" * bar_len
        print(f"│  step {i:2d}: x={trajectory[i]:6.2f} loss={losses[i]:7.3f} {bar}│")
    
    if len(losses) > 15:
        print(f"│  ... ({len(losses)-15} steps 생략)           │")
    print(f"└──────────────────────────────────────────────┘")

lr_slider = widgets.FloatSlider(value=0.1, min=0.01, max=0.9, step=0.01, description='학습률:')
step_slider = widgets.IntSlider(value=20, min=5, max=100, step=5, description='스텝 수:')
widgets.interact(gradient_descent_simulator, lr=lr_slider, steps=step_slider)

In [ ]:
# ── 위젯 5: 활성화 함수 비교기 ──
def activation_comparator(func_name, x_val):
    funcs = {
        'Sigmoid': sigmoid,
        'Tanh': tanh_func,
        'ReLU': relu,
        'Leaky ReLU': leaky_relu,
        'GELU': gelu
    }
    f = funcs[func_name]
    
    x_range = np.linspace(-5, 5, 51)
    y_range = f(x_range)
    
    clear_output(wait=True)
    print(f"┌──────────────────────────────────────────────┐")
    print(f"│  활성화 함수: {func_name:<20}               │")
    print(f"│  x = {x_val:.1f}  →  f(x) = {f(np.array([x_val]))[0]:.4f}  │")
    print(f"├──────────────────────────────────────────────┤")
    
    # ASCII 그래프
    for row in range(10, -11, -1):
        y_level = row * 0.2
        line = "│  "
        for col in range(51):
            x_pos = x_range[col]
            y_pos = y_range[col]
            if abs(x_pos - x_val) < 0.15:
                line += "●"  # 현재 위치
            elif abs(y_pos - y_level) < 0.15:
                line += "─"
            elif abs(x_pos) < 0.05:
                line += "│"
            elif abs(y_level) < 0.05:
                line += "─"
            else:
                line += " "
        print(line + "│")
    
    print(f"└──────────────────────────────────────────────┘")

func_dd = widgets.Dropdown(options=['Sigmoid', 'Tanh', 'ReLU', 'Leaky ReLU', 'GELU'],
                           description='함수:')
x_slider = widgets.FloatSlider(value=0, min=-5, max=5, step=0.5, description='x 값:')
widgets.interact(activation_comparator, func_name=func_dd, x_val=x_slider)

---

## 4.8 신경망 기초 **연습문제** (심화)

### 문제 11. 퍼셉트론 구현

In [ ]:
# NAND 게이트를 퍼셉트론으로 구현하라.
# NAND: NOT AND → (0,0)→1, (0,1)→1, (1,0)→1, (1,1)→0
# 적절한 weights와 bias를 찾아라.

# 여기에 코드를 작성하라

### 문제 12. 활성화 함수 비교

In [ ]:
# z = [-3, -2, -1, 0, 1, 2, 3] 에 대해
# Sigmoid, ReLU, GELU의 출력값을 비교하는 표를 출력하라.
# 또한 세 함수의 차이점을 한 줄로 서술하라.

# 여기에 코드를 작성하라

### 문제 13. 경사 하강법 실험

In [ ]:
# f(x) = x⁴ - 3x³ + 2 의 최솟값을 경사 하강법으로 찾아라.
# 미분: f'(x) = 4x³ - 9x²
# 학습률 0.01로 100스텝을 실행하고, x와 loss의 변화를 추적하라.
# 시작점: x = 4.0

# 여기에 코드를 작성하라

### 문제 14. 순전파 계산

In [ ]:
# 다음 2층 신경망의 순전파를 직접 계산하라.
# 입력: x = [1.0, 0.5]
# 층 1: W1 = [[0.1, 0.3], [0.2, 0.4]], b1 = [0.1, 0.1], 활성화: ReLU
# 층 2: W2 = [[0.5], [0.6]], b2 = [0.2], 활성화: Sigmoid
# 최종 출력을 계산하라.

# 여기에 코드를 작성하라

### 문제 15. MSE vs CE 비교 (보너스)

In [ ]:
# 정답: y_true = [1, 0, 1]
# 예측 A: y_pred_A = [0.9, 0.2, 0.8]  (좋은 예측)
# 예측 B: y_pred_B = [0.6, 0.4, 0.6]  (애매한 예측)
# 예측 C: y_pred_C = [0.1, 0.9, 0.2]  (나쁜 예측)
# 
# 각각에 대해 MSE와 Cross-Entropy를 계산하고,
# 어떤 손실 함수가 "틀린 예측"에 더 강한 벌점을 주는지 분석하라.

# 여기에 코드를 작성하라

<details>
<summary>▶ 신경망 기초 연습문제 해답 (클릭하여 펼치기)</summary>

#### 해답 11

<pre><code class="python">def perceptron(inputs, weights, bias):
    z = np.dot(inputs, weights) + bias
    return 1 if z &gt;= 0 else 0

# NAND = NOT AND → 모든 가중치를 음수로, bias를 양수로
weights = np.array([-0.5, -0.5])
bias = 0.7

print("NAND 게이트:")
for x1, x2 in [(0,0), (0,1), (1,0), (1,1)]:
    out = perceptron(np.array([x1, x2]), weights, bias)
    expected = 0 if (x1 and x2) else 1
    print(f"  ({x1}, {x2}) → {out}  {'✅' if out == expected else '❌'}")</code></pre>

#### 해답 12

<pre><code class="python">z = np.array([-3, -2, -1, 0, 1, 2, 3])
print(f"{'z':&gt;4} {'Sigmoid':&gt;10} {'ReLU':&gt;10} {'GELU':&gt;10}")
print(f"{'-'*34}")
for val in z:
    v = np.array([val], dtype=float)
    print(f"{val:4d} {sigmoid(v)[0]:10.4f} {relu(v)[0]:10.4f} {gelu(v)[0]:10.4f}")
print()
print("차이점: Sigmoid는 0~1로 압축, ReLU는 음수를 0으로 버림, GELU는 음수를 부드럽게 감쇄한다.")</code></pre>

#### 해답 13

<pre><code class="python">def f(x):
    return x**4 - 3*x**3 + 2
def df(x):
    return 4*x**3 - 9*x**2

x = 4.0
lr = 0.01
print(f"{'step':&gt;4} {'x':&gt;10} {'f(x)':&gt;10}")
for step in range(100):
    if step % 10 == 0:
        print(f"{step:4d} {x:10.4f} {f(x):10.4f}")
    x = x - lr * df(x)
print(f"{'최종':&gt;4} {x:10.4f} {f(x):10.4f}")</code></pre>

#### 해답 14

<pre><code class="python">x = np.array([1.0, 0.5])
W1 = np.array([[0.1, 0.3], [0.2, 0.4]])
b1 = np.array([0.1, 0.1])
W2 = np.array([[0.5], [0.6]])
b2 = np.array([0.2])

# 층 1
z1 = x @ W1 + b1
a1 = relu(z1)
print(f"z1 = {z1}, a1(ReLU) = {a1}")

# 층 2
z2 = a1 @ W2 + b2
output = sigmoid(z2)
print(f"z2 = {z2}, output(Sigmoid) = {output}")</code></pre>

#### 해답 15

<pre><code class="python">y_true = np.array([1, 0, 1])
preds = {
    'A(좋은)': np.array([0.9, 0.2, 0.8]),
    'B(애매)': np.array([0.6, 0.4, 0.6]),
    'C(나쁜)': np.array([0.1, 0.9, 0.2])
}

for name, pred in preds.items():
    mse = np.mean((y_true - pred)**2)
    ce = cross_entropy_loss(y_true, pred)
    print(f"  예측 {name}: MSE={mse:.4f}, CE={ce:.4f}")

print()
print("[분석] CE가 MSE보다 틀린 예측에 훨씬 강한 벌점을 부여한다.")
print("       예측 C에서 CE는 폭발적으로 증가하지만, MSE는 선형적으로만 증가한다.")
print("       → 분류 문제에서는 Cross-Entropy가 더 적합하다.")</code></pre>

</details>

---

# 종합 정리

```
┌─────────────────────────────────────────────────────────────────┐
│  4주차 핵심 요약                                                 │
├──────────────┬──────────────────────────────────────────────────┤
│  PART 1      │ NumPy: 벡터화, 브로드캐스팅, 불리언 인덱싱,      │
│  NumPy       │ 통계 함수, 행렬 연산, 메모리 구조                 │
├──────────────┼──────────────────────────────────────────────────┤
│  PART 2      │ Pandas: DataFrame/Series, CSV, 필터링,            │
│  Pandas      │ groupby, 결측값 처리                              │
├──────────────┼──────────────────────────────────────────────────┤
│  PART 3      │ 프로젝트: 에어비앤비 7단계 분석 파이프라인        │
│  프로젝트    │ 필터링 → 결측값 → groupby → 이상치 → 결론        │
├──────────────┼──────────────────────────────────────────────────┤
│  PART 4      │ 신경망: 퍼셉트론, 활성화 함수 6종,               │
│  신경망 기초 │ 손실 함수(MSE/CE), 경사 하강법, 순전파, 역전파    │
└──────────────┴──────────────────────────────────────────────────┘
```

### → AI 연결: 다음 주차 예고

> 5주차에서는 오늘 분석한 에어비앤비 데이터를 **시각화**한다.
> Matplotlib, Seaborn, Plotly로 히스토그램, 산점도, 박스플롯, 히트맵을 그리고,
> **"왜 맨해튼이 비싼가?"**에 대한 시각적 증거를 찾는다.
> 오늘의 숫자(groupby 결과)가 내일의 그래프가 된다!

---

*4주차 끝 — PART 2: 데이터 분석 기초의 시작*